In [37]:
import random
import math


**Module Base Class**

In [38]:
class Module:
    def __init__(self):
        self.training = True
        
    def forward(self,x):
        raise NotImplementedError
    
    def backward(self,grad):
        raise NotImplementedError
    
    def parameter(self):
        return []
    
    def train(self):
        self.training= True
    def evals(self):
        self.training =False
        

    
    

**Linear Layer**

In [39]:
class Linear(Module):
    def __init__(self,fan_in,fan_out):
        super().__init__()
        std = math.sqrt(2/fan_in)
        self.weights = [[random.gauss(0,std) for _ in range(fan_in)] for _ in range(fan_out) ]
        self.bias = [0.0]*fan_out
        self.weights_grad = [[0.0]*fan_in  for _ in range(fan_out)]
        self.bias_grad = [0.0]*fan_out
        
        self.fan_in = fan_in
        self.fan_out = fan_out
        self.input =None
    
    def forward(self, x):
        self.input=x
        output =[]
        for i in range(self.fan_out):
            val = self.bias[i]
            for j in range(self.fan_in):
                val+=self.weights[i][j]*x[j]
            output.append(val)
        return output
    
    
    def backward(self, grad):
        input_grad =[0.0]*self.fan_in
        for i in range(self.fan_out):
            self.bias_grad[i]+=grad[i]
            for j in range(self.fan_in):
                self.weights_grad[i][j]+=grad[i]*self.input[j]
                input_grad[i]+= grad[i]*self.weights[i][j]
        return input_grad
    def parameter(self):
        params =[]
        for i in range(self.fan_out):
            for j in range(self.fan_in):
                params.append((self.weights,i,j,self.weights_grad))
            params.append((self.bias,i,None,self.bias_grad))
        return params
    
    
                
        
        

**Activation Module**

In [40]:
class Relu(Module):
    def __init__(self):
        super().__init__()
        self.mask =None
    
    def forward(self, x):
        self.mask=[1.0 if v>0 else 0 for v in x]
        return [max(0,v) for v in x]
    
    def backward(self, grad):
        return [g*m for (g,m) in zip(grad,self.mask)]
    
    
class Sigmoid(Module):
    def __init__(self):
        super().__init__()
        self.output =None
    
    def forward(self, x):
        self.output =[]
        for v in x:
            v=max(-500,min(500,v))
            self.output.append(1/(1+math.exp(-v)))
        return self.output
    
    
    def backward(self, grad):
        return [g*o for (g,o) in zip(grad,self.output)]
    

    
class Tanh(Module):
    def __init__(self):
        super().__init__()
        self.output =None
    
    def forward(self, x):
        self.output =[]
        for v in x:
            
            self.output.append(math.tanh(v))
        return self.output
    
    
    def backward(self, grad):
        return [g*(1-o*o) for (g,o) in zip(grad,self.output)]
    
    

In [41]:
class Dropout(Module):
    def __init__(self,p):
        super().__init__()
        self.p =p
        self.mask =None
    
    def forward(self, x):
        if not self.training:
            return x
        self.mask =[0.0 if random.random()<self.p else 1.0 /(1-self.p) for _ in x]
        return [v * m for v, m in zip(x, self.mask)]
    
    def backward(self, grad):
        if self.mask is None:
            return grad
        return [g*m for (g,m) in zip(grad,self.mask)]


In [42]:
class BatchNorm(Module):
    def __init__(self,size,momentum=0.1,eps=1e-15):
        super().__init__()
        self.gamma = [1.0] * size
        self.beta = [0.0] * size
        self.gamma_grads = [0.0] * size
        self.beta_grads = [0.0] * size
        self.running_mean = [0.0] * size
        self.running_var = [1.0] * size
        self.momentum = momentum
        self.eps = eps
        self.x_norm = None
        self.std_inv = None
        self.batch_input = None
    
    def forward(self,batch):
        bathc_size =len(batch)
        
        if self.training:
            mean=[]
            
            ##not completed

In [43]:
class Sequential(Module):
    def __init__(self,*module):
        super().__init__()
        self.modules = list(module)
    
    def forward(self, x):
        for module in self.modules:
            x = module.forward(x)
        return x
    
    def backward(self, grad):
        for module in reversed(self.modules):
            grad = module.backward(grad)
            
        return grad 
    def parameters(self):
        params =[]
        for module in self.modules:
            params.extend(module.parameter())
        return params
    
    def train(self):
        self.training = True
        for module in self.modules:
            module.train()
    def evals(self):
        self.training =False
        for module in self.modules:
            module.evals()
            
        

In [44]:
class MSELoss:
    def __call__(self, predicted, target):
        self.predicted = predicted
        self.target = target
        n = len(predicted)
        self.loss = sum((p - t) ** 2 for p, t in zip(predicted, target)) / n
        return self.loss

    def backward(self):
        n = len(self.predicted)
        return [2 * (p - t) / n for p, t in zip(self.predicted, self.target)]


class BCELoss:
    def __call__(self, predicted, target):
        self.predicted = predicted
        self.target = target
        eps = 1e-7
        n = len(predicted)
        self.loss = 0
        for p, t in zip(predicted, target):
            p = max(eps, min(1 - eps, p))
            self.loss += -(t * math.log(p) + (1 - t) * math.log(1 - p))
        self.loss /= n
        return self.loss

    def backward(self):
        eps = 1e-7
        n = len(self.predicted)
        grads = []
        for p, t in zip(self.predicted, self.target):
            p = max(eps, min(1 - eps, p))
            grads.append((-t / p + (1 - t) / (1 - p)) / n)
        return grads

In [45]:
class SGD:
    def __init__(self, parameters, lr=0.01):
        self.params = parameters
        self.lr = lr

    def step(self):
        for container, i, j, grad_container in self.params:
            if j is not None:
                container[i][j] -= self.lr * grad_container[i][j]
            else:
                container[i] -= self.lr * grad_container[i]

    def zero_grad(self):
        for container, i, j, grad_container in self.params:
            if j is not None:
                grad_container[i][j] = 0.0
            else:
                grad_container[i] = 0.0


class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = parameters
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = [0.0] * len(parameters)
        self.v = [0.0] * len(parameters)

    def step(self):
        self.t += 1
        for idx, (container, i, j, grad_container) in enumerate(self.params):
            if j is not None:
                g = grad_container[i][j]
            else:
                g = grad_container[i]

            self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * g
            self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * g * g

            m_hat = self.m[idx] / (1 - self.beta1 ** self.t)
            v_hat = self.v[idx] / (1 - self.beta2 ** self.t)

            update = self.lr * m_hat / (math.sqrt(v_hat) + self.eps)

            if j is not None:
                container[i][j] -= update
            else:
                container[i] -= update

    def zero_grad(self):
        for container, i, j, grad_container in self.params:
            if j is not None:
                grad_container[i][j] = 0.0
            else:
                grad_container[i] = 0.0

In [46]:
class Dataloader():
    def __init__(self,data,batch_size=32,shuffle=True):
        self.data=data
        self.batch_size =batch_size
        self.shuffle =shuffle
        
    def __iter__(self):
        indices =list(range(len(self.data)))
        if self.shuffle:
            random.shuffle(indices)
        for start in range(0,len(indices),self.batch_size):
            batch_indices=indices[start:start*self.batch_size]
            batch = [self.data[i] for i in batch_indices]
            inputs = [item[0] for item in batch]
            target = [item[1] for item in batch]
            yield inputs,target
            
    def __len__(self):
        return (len(self.data) + self.batch_size - 1) // self.batch_size
        
        

In [47]:
def make_circle_data(n=500, seed=42):
    random.seed(seed)
    data = []
    for _ in range(n):
        x = random.uniform(-2, 2)
        y = random.uniform(-2, 2)
        label = 1.0 if x * x + y * y < 1.5 else 0.0
        data.append(([x, y], [label]))
    return data


def train():
    random.seed(42)

    model = Sequential(
        Linear(2, 16),
        Relu(),
        Linear(16, 16),
        Relu(),
        Linear(16, 8),
        Relu(),
        Linear(8, 1),
        Sigmoid(),
    )

    criterion = BCELoss()
    optimizer = Adam(model.parameters(), lr=0.01)

    data = make_circle_data(500)
    split = int(len(data) * 0.8)
    train_data = data[:split]
    test_data = data[split:]

    loader = Dataloader(train_data, batch_size=16, shuffle=True)

    model.train()

    for epoch in range(100):
        total_loss = 0
        total_correct = 0
        total_samples = 0

        for batch_inputs, batch_targets in loader:
            batch_loss = 0
            for x, t in zip(batch_inputs, batch_targets):
                pred = model.forward(x)
                loss = criterion(pred, t)
                batch_loss += loss

                optimizer.zero_grad()
                grad = criterion.backward()
                model.backward(grad)
                optimizer.step()

                predicted_class = 1.0 if pred[0] >= 0.5 else 0.0
                if predicted_class == t[0]:
                    total_correct += 1
                total_samples += 1

            total_loss += batch_loss

        avg_loss = total_loss / total_samples
        accuracy = total_correct / total_samples * 100

        if epoch % 10 == 0 or epoch == 99:
            print(f"Epoch {epoch:3d} | Loss: {avg_loss:.6f} | Train Accuracy: {accuracy:.1f}%")

    model.eval()
    correct = 0
    for x, t in test_data:
        pred = model.forward(x)
        predicted_class = 1.0 if pred[0] >= 0.5 else 0.0
        if predicted_class == t[0]:
            correct += 1
    test_accuracy = correct / len(test_data) * 100
    print(f"\nTest Accuracy: {test_accuracy:.1f}% ({correct}/{len(test_data)})")

    return model, test_accuracy

In [48]:
train()

IndexError: list index out of range